# Produção · previsão SEMANAL (1 a 4 semanas à frente)

Roda os **5 melhores modelos semanais** (MAE médio no teste 2022–2025) a partir da **última semana completa**. A última semana dos dados chega incompleta (atraso de notificação: em 03/2026 tinha ~36% dos casos), então por padrão ela é descartada como origem (`DESCARTAR = 1`) e o h = 1 a re-estima:

| # | Modelo | MAE h = 1 / 2 / 3 / 4 (casos/semana) |
|---|---|---|
| 1 | **ENS_stack_sem** (campeão) | 270 / 383 / 475 / 583 |
| 2 | ARIMA_Fourier_sem | 252 / 398 / 525 / 646 |
| 3 | GBM_xgb_optuna_sem | 308 / 403 / 508 / 639 |
| 4 | ENS_GBM_ARIMA_sem | 271 / 414 / 526 / 648 |
| 5 | ENS_GBM_ARIMA_MTGNNcresc_sem | 267 / 418 / 544 / 680 |

O campeão é a média ponderada de **9 membros** (XGBoost em 5 versões, ARIMA + Fourier, 2 MTGNN e Chronos-2 ajustado). Cada membro é **retreinado** com os dados até a semana de origem, com os hiperparâmetros e a seleção de variáveis da pesquisa (`parametros/`). Os pesos (horizonte × fase da UF) vêm da afinação 2018–2021.

**Ambiente:** GPU (L4 ou T4). Sem GPU funciona, mas as redes e o Chronos ficam lentos; para um teste rápido, use `SO_MEMBROS` (ex.: só ARIMA e XGBoost). Se um membro falhar, os pesos do stacking são renormalizados entre os que rodaram.

**Antes:** monte a área de trabalho (`python scripts/04_montar_area_de_trabalho.py`) e suba a pasta para `MyDrive/dengue_V4/`.

**Saídas** em `producao_semanal/`: um arquivo por membro, `previsao_semanal_<data>.parquet` (top 5, 9 quantis) e `tabela_semanal_<data>.csv` (campeão, por UF).


In [ ]:
PACOTES = 'tsfresh catboost xgboost lightgbm statsmodels "chronos-forecasting>=2.0" peft && pip uninstall -y -q torchao'

# @title 1 · Área de trabalho (Drive no Colab ou pasta local), pacotes e contexto
import os, sys, shutil, glob, json, time, subprocess, traceback
import numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_DRIVE, PASTA_LOCAL, NO_COLAB = '/content/drive/MyDrive/dengue_V4', '/content/prod', True
except ImportError:
    PASTA_DRIVE = PASTA_LOCAL = os.path.abspath('.'); NO_COLAB = False          # Jupyter aberto na área de trabalho
if NO_COLAB:
    if PACOTES:
        subprocess.run(f'pip install -q {PACOTES}', shell=True, check=True)
    os.makedirs(PASTA_LOCAL, exist_ok=True)
    for item in ('v4_lib.py', 'comp_lib.py', 'comp_modelos.py', 'uf_lib.py', 'semanal_lib.py', 'r2_lib.py', 'r3_lib.py', 'r4_lib.py',
                 'r5_lib.py', 'r6_lib.py', 'r7_lib.py', 'prod_lib.py', 'relatorio_campeoes.py', 'graficos_previsoes.py',
                 'dados', 'dados_uf', 'referencias_imdc3', 'parametros'):
        o, d = os.path.join(PASTA_DRIVE, item), os.path.join(PASTA_LOCAL, item)
        assert os.path.exists(o), f'Faltou {item} em {PASTA_DRIVE}: rode scripts/04_montar_area_de_trabalho.py e suba a pasta'
        shutil.copytree(o, d, dirs_exist_ok=True) if os.path.isdir(o) else shutil.copy(o, d)
sys.path.insert(0, PASTA_LOCAL)
import importlib, v4_lib as V, comp_lib as L, uf_lib as U, semanal_lib as S, r2_lib as R2, r3_lib as R3, prod_lib as PL
for m in (V, L, U, S, R2, R3, PL):
    importlib.reload(m)
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30); pd.set_option('display.max_rows', 120)
ctx = U.ContextoUF(os.path.join(PASTA_LOCAL, 'dados'), os.path.join(PASTA_LOCAL, 'referencias_imdc3'), os.path.join(PASTA_LOCAL, 'dados_uf'))
D = ctx.Du
TUNING, TESTE = U.TUNING, U.TESTE
P = {k: os.path.join(PASTA_DRIVE, v) for k, v in dict(PS='producao_semanal', PT='producao_temporada', REL='relatorio_campeoes',
                                                      CACHE='cache_producao').items()}
for p in P.values():
    os.makedirs(p, exist_ok=True)
PAR = PL.ler_parametros(os.path.join(PASTA_LOCAL, 'parametros'))
ULTIMA = pd.Timestamp(D.datas[D.ultimo_obs]).date()
print('última semana observada:', ULTIMA, '· UFs:', D.N)


In [ ]:
# @title 2 · Rodada de produção e membros
SO_MEMBROS = None          # ex.: ['ARIMA_Fourier_sem', 'GBM_xgb_todas_sem'] para um teste rápido
DESCARTAR = 1              # últimas semanas incompletas (atraso de notificação); 0 = usar a última semana como origem
ROD = PL.rodada_producao(D, descartar=DESCARTAR)   # origem = última semana completa; h = 1 re-estima a incompleta
print(ROD)
MEMBROS = PL.membros_semanais(ctx, PAR, P['CACHE'], so=SO_MEMBROS)
print('membros:', [m.nome for m in MEMBROS])


In [ ]:
# @title 3 · Treinar, prever e combinar (retoma de onde parou)
PM, ERROS = PL.prever_semanal(ctx, MEMBROS, os.path.join(P['PS'], 'membros'), ROD)
print('erros:', ERROS)
PREV = PL.ensembles_semanais(PM, D, PAR)
PREV = PREV[PREV.modelo.isin(PL.TOP5_SEMANAL + [m.nome for m in MEMBROS])]
rot = f'{ULTIMA:%Y%m%d}'
PL.gravar(PREV, P['PS'], f'previsao_semanal_{rot}.parquet'); PL.gravar(PREV, P['PS'], 'previsao_semanal_ultimo.parquet')
TAB = PL.tabela_semanal(PREV, D)
PL.gravar(TAB, P['PS'], f'tabela_semanal_{rot}.csv'); PL.gravar(TAB, P['PS'], 'tabela_semanal_ultimo.csv')
ORIGEM = pd.Timestamp(D.datas[list(ROD.values())[0]['origens'][0]]).date()
print(f'Campeão ({PL.CAMPEAO_SEMANAL}) · origem {ORIGEM} (última semana completa) · casos previstos (mediana) por semana-alvo:')
TAB.pivot_table(index='uf', columns='date', values='pred').round(0)


In [ ]:
# @title 4 · (opcional) Reproduzir o teste 2022–2025 com o mesmo código (horas de GPU)
RODAR_BACKTEST = False
if RODAR_BACKTEST:
    ROD_T = S.rodadas(D, temporadas=TESTE, felipe=False)
    PB, _ = PL.prever_semanal(ctx, PL.membros_semanais(ctx, PAR, P['CACHE']), os.path.join(P['PS'], 'backtest'), ROD_T)
    EB = S.com_verdade(PL.ensembles_semanais(PB, D, PAR), D)
    RES = pd.DataFrame([dict(modelo=m, h=h, **S.metricas(g)) for (m, h), g in EB.groupby(['modelo', 'h'])])
    PL.gravar(RES, P['PS'], 'backtest_metricas.csv')
    display(RES[RES.modelo.isin(PL.TOP5_SEMANAL)].pivot_table(index='modelo', columns='h', values='MAE').round(0))
